# Lecture 24 · A Clock Inside: the notebook

*The Anatomy of AI.* In the lecture a tiny network learned to add on a clock of 113 hours. It memorised its homework at once, looked hopeless on new sums for six thousand steps, then suddenly got them all right; inside, it had built a clock out of waves. Here you train that network yourself and read its clock:

1. **Adding on a clock**, and the 12,769 sums (Plates I, III, IV).
2. **The network, written by hand** (Plate V).
3. **Train it and watch it grok** (Plates VI, VIII), then without weight decay (Plate VII).
4. **Waves over the numbers** (Plate IX) and **the network's favourite waves** (Plate X).
5. **The clock** (Plates XI, XII).
6. **One neuron, two waves** (Plate XIII) and **every neuron has a frequency** (Plate XIV).
7. **The vote** (Plate XV).
8. **Grokking from the inside**: progress measures (Plate XVI).

Every key number is checked against the lecture's own logs with `check()`. It runs on a free Colab **T4 GPU** in about 13 minutes, most of it the training. No downloads, no keys.

In [ ]:
# Pinned installs. Free Colab already has torch; a machine without it gets the pinned version.
import importlib.util, subprocess, sys
def pip(*args):                                   # quiet: pip's output is shown only if an install fails
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *args],
                       capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stdout[-3000:] + r.stderr[-3000:])
if importlib.util.find_spec("torch") is None:
    pip("torch==2.5.1")
pip("numpy==2.0.2", "matplotlib==3.10.0")

In [ ]:
import io, json, math, os, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)      # library notices, not results
warnings.filterwarnings("ignore", category=UserWarning)
import numpy as np
import matplotlib.pyplot as plt
import torch

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("running on", DEV)
torch.backends.cuda.matmul.allow_tf32 = False                  # plain float32 everywhere, as the lecture ran
T0 = time.time()

# The course's look: black ink on paper, no colour.
PAPER, INK, INK2 = "#F1EBDD", "#16130F", "#7A7062"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "font.family": "serif", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "hatch.color": INK, "hatch.linewidth": 0.6, "figure.dpi": 110, "legend.frameon": False,
    "lines.color": INK, "patch.edgecolor": INK, "axes.prop_cycle": plt.cycler(color=[INK, INK2]),
})


def minus(text):
    """Print negative numbers with a real minus sign, as the lecture does."""
    return str(text).replace("-", "\u2212")


def check(name, value, expected, tol=0):
    """Compare a result with the lecture's own number. Prints ✓, or stops the notebook with ✗."""
    number = isinstance(expected, (int, float)) and not isinstance(expected, bool)
    if number:
        ok = abs(float(value) - expected) <= tol
        whole = isinstance(expected, int) and float(value) == int(value)
        shown = f"{int(value):,}" if whole else f"{float(value):.6g}"
        said = f"{expected:,}" if isinstance(expected, int) else expected
    else:
        ok = value == expected
        shown, said = repr(value), repr(expected)
    extra = f" ± {tol:g}" if number and tol else ""
    print(f"{'✓' if ok else '✗'} {name}: {minus(shown)}   (lecture: {minus(said)}{extra})")
    if not ok:
        raise AssertionError(f"{name}: got {shown}, the lecture says {expected}")


def fetch(url, path, what):
    """Download a file once (quietly), and say in one line what came down."""
    import requests
    if not os.path.exists(path):
        t = time.time()
        with requests.get(url, stream=True, timeout=300) as r:
            r.raise_for_status()
            with open(path + ".part", "wb") as f:
                for chunk in r.iter_content(1 << 22):
                    f.write(chunk)
        os.rename(path + ".part", path)
        print(f"downloaded {what} ({os.path.getsize(path) / 1e6:.1f} MB) in {time.time() - t:.0f} s")
    return path
T0_START = time.time()

The lecture's own numbers, from its logs (`lectures/24-a-clock-inside/experiments/raw/`).

In [ ]:
LECTURE = {
 "n_train": 3830,
 "split_sha256": "849d3293744dd77cf81d95b25526b12d6738be9e4faf1f06b19bed27cb04049c",
 "n_params": 226688,
 "log": {
  "0": [
   4.760717,
   4.756662,
   0.01227,
   0.00973,
   42.2131
  ],
  "200": [
   0.019126,
   16.413635,
   1.0,
   0.01712,
   60.2984
  ],
  "1400": [
   0.0,
   26.287709,
   1.0,
   0.03423,
   57.7273
  ],
  "4000": [
   0.0,
   19.436927,
   1.0,
   0.07294,
   50.9206
  ],
  "6000": [
   0.0,
   9.269061,
   1.0,
   0.24141,
   47.4727
  ],
  "7500": [
   0.0,
   0.021585,
   1.0,
   0.99217,
   39.3932
  ],
  "8000": [
   0.0,
   1.9e-05,
   1.0,
   1.0,
   35.3773
  ],
  "40000": [
   3e-06,
   5e-06,
   1.0,
   1.0,
   32.3244
  ]
 },
 "cols": [
  "train_loss",
  "test_loss",
  "train_acc",
  "test_acc",
  "weight_norm"
 ],
 "step_train99": 200,
 "step_test99": 7500,
 "norm_peak": 60.2984,
 "test_loss_peak": 26.287709,
 "train_seconds_on_T4": 420.1,
 "no_decay_10000": [
  0.0,
  52.372963,
  1.0,
  0.02786,
  73.6917
 ],
 "answer_at": {
  "0": 68,
  "1000": 22,
  "4000": 0,
  "7000": 0,
  "8000": 16
 },
 "p16_at_7000": 0.00075,
 "p16": 0.9999997,
 "waves_at_16": {
  "1": 12.207,
  "8": 33.175,
  "21": 19.709
 },
 "vote_16": 65.091,
 "vote_best_wrong": 49.877,
 "logit_16": 67.138,
 "logit_best_wrong": 51.394,
 "neuron_182_fires": 2.2536,
 "key": [
  1,
  8,
  21
 ],
 "neuron_counts": [
  103,
  188,
  221
 ],
 "purity_median": 0.9508,
 "logit_frac_key": 0.95183,
 "embed_norms": {
  "1": 5.7679,
  "8": 7.1022,
  "21": 7.5561
 },
 "embed_norm_other_max": 1.9561,
 "clock_fit": {
  "1": 0.9989,
  "8": 0.9993,
  "21": 0.9971
 },
 "neuron_freq": {
  "182": 1,
  "27": 8,
  "357": 21
 },
 "neuron_shares": {
  "182": [
   0.1316,
   0.7756
  ],
  "27": [
   0.1673,
   0.7587
  ],
  "357": [
   0.1138,
   0.7783
  ]
 },
 "progress": {
  "2000": {
   "test_loss": 24.92169,
   "restricted_test": 1.84534,
   "excluded_train": 0.00028,
   "logit_frac_key": 0.05243
  },
  "4000": {
   "test_loss": 19.43693,
   "restricted_test": 0.73173,
   "excluded_train": 0.41462,
   "logit_frac_key": 0.2422
  },
  "6000": {
   "test_loss": 9.26906,
   "restricted_test": 0.01278,
   "excluded_train": 7.40154,
   "logit_frac_key": 0.66572
  },
  "8000": {
   "test_loss": 2e-05,
   "restricted_test": 0.0,
   "excluded_train": 15.73217,
   "logit_frac_key": 0.95768
  }
 },
 "grok_step_wd3": 1800
}

## Part 1 · Adding on a clock (Plates I, III, IV)

On a clock of 113 hours, 37 + 92 = 129 wraps round to 16: (37 + 92) mod 113 = 16. There are 113 × 113 = 12,769 sums. The network sees 30% of them with their answers (the *training* sums, its homework) and is tested on the other 70% (the *test* sums, the exam). The lecture's seed decides which; with the same seed you get the same split, bit for bit.

In [ ]:
import hashlib
P = 113
print("37 + 92 =", 37 + 92, "->", (37 + 92) % P, "on a clock of 113")

def all_pairs():
    a = torch.arange(P, device=DEV).repeat_interleave(P)
    b = torch.arange(P, device=DEV).repeat(P)
    return torch.stack([a, b, torch.full_like(a, P)], 1), (a + b) % P      # the tokens "a b =", and the answers

X, Y = all_pairs()
perm = torch.randperm(P * P, generator=torch.Generator().manual_seed(0)).to(DEV)
n_train = int(0.3 * P * P)
TR, TE = perm[:n_train], perm[n_train:]
is_train = torch.zeros(P * P, dtype=torch.bool); is_train[TR.cpu()] = True
bits = "".join("1" if v else "0" for v in is_train.tolist())
print(f"{n_train:,} training sums, {P * P - n_train:,} test sums; 37 + 92 is a {'training' if is_train[37 * P + 92] else 'test'} sum")
check("training sums", n_train, LECTURE["n_train"])
check("the split is the lecture's, bit for bit (sha256)", hashlib.sha256(bits.encode()).hexdigest(), LECTURE["split_sha256"])

fig, ax = plt.subplots(figsize=(3.6, 3.6))
ax.imshow(is_train.reshape(P, P).numpy(), cmap="Greys", vmin=0, vmax=1.6, interpolation="nearest")
ax.set_xlabel("b"); ax.set_ylabel("a"); ax.set_title("dark: the 3,830 training sums", fontsize=10)
ax.plot(92, 37, marker="o", ms=6, mfc=PAPER, mec=INK); ax.annotate("37 + 92", (92, 37), (60, 12), fontsize=8)
plt.show()

## Part 2 · The network, written by hand (Plate V)

One transformer block, Lecture 20's, with nothing extra:
- three tokens, `a`, `b` and `=`, each turned into 128 numbers by a learned table (plus a learned position);
- four attention heads mixing the three columns;
- 512 neurons with ReLU (Lecture 1's), added back;
- 113 scores read off the last column, one per possible answer.

There is no LayerNorm. The weights start random, drawn from the lecture's seed.

In [ ]:
def init_params(seed, d=128, heads=4, d_mlp=512):
    g = torch.Generator().manual_seed(seed)
    r = lambda *shape, scale: (torch.randn(*shape, generator=g) * scale).to(DEV).requires_grad_()
    z = lambda *shape: torch.zeros(*shape, device=DEV, requires_grad=True)
    dh = d // heads
    return {"W_E": r(P + 1, d, scale=d ** -0.5), "W_pos": r(3, d, scale=d ** -0.5),        # 113 numbers + "="
            "W_Q": r(heads, d, dh, scale=d ** -0.5), "W_K": r(heads, d, dh, scale=d ** -0.5),
            "W_V": r(heads, d, dh, scale=d ** -0.5), "W_O": r(heads, dh, d, scale=d ** -0.5),
            "W_in": r(d, d_mlp, scale=d ** -0.5), "b_in": z(d_mlp),
            "W_out": r(d_mlp, d, scale=d ** -0.5), "b_out": z(d), "W_U": r(d, P, scale=(P + 1) ** -0.5)}

def forward(W, x, keep=None):
    """x: [B, 3] tokens. Returns the 113 scores at "=". keep: a dict to fill with the neurons' activations."""
    h = W["W_E"][x] + W["W_pos"]
    q, k, v = (torch.einsum("btd,hde->bhte", h, W[n]) for n in ("W_Q", "W_K", "W_V"))
    s = q @ k.transpose(-1, -2) / q.shape[-1] ** 0.5
    s = s.masked_fill(torch.triu(torch.ones(3, 3, dtype=torch.bool, device=x.device), 1), float("-inf"))
    h = h + torch.einsum("bhte,hed->btd", s.softmax(-1) @ v, W["W_O"])
    post = torch.relu(h @ W["W_in"] + W["b_in"])
    h = h + post @ W["W_out"] + W["b_out"]
    if keep is not None:
        keep["post"] = post[:, -1]
    return h[:, -1] @ W["W_U"]

W = init_params(0)
print(f"{sum(v.numel() for v in W.values()):,} weights")
check("weights in the network", sum(v.numel() for v in W.values()), LECTURE["n_params"])

## Part 3 · Train it and watch it grok (Plates VI, VII, VIII)

The lecture's recipe (after Nanda et al. 2023): AdamW with step size 0.001, a 10-step warm-up, all 3,830 training sums at every step, 40,000 steps, **weight decay 1** (a gentle pull of every weight toward zero). Every 100 steps we score the training and the test sums and measure the size of all the weights. About 8 minutes on a T4.

In [ ]:
def train(wd, seed=0, steps=40000, every=100, keep_every=500, keep_until=14000):
    torch.manual_seed(seed)
    W = init_params(seed)
    opt = torch.optim.AdamW(W.values(), lr=1e-3, weight_decay=wd, betas=(0.9, 0.98))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(s / 10, 1))
    log, snaps, answers = [], {}, {}
    x3792 = torch.tensor([[37, 92, P]], device=DEV)
    def loss_acc(idx):
        lg = forward(W, X[idx])
        return float(-lg.double().log_softmax(-1).gather(1, Y[idx, None]).mean()), float((lg.argmax(-1) == Y[idx]).float().mean())
    for step in range(steps + 1):
        if step % every == 0:
            with torch.no_grad():
                (trl, tra), (tel, tea) = loss_acc(TR), loss_acc(TE)
                norm = float(sum((w.double() ** 2).sum() for w in W.values()) ** 0.5)
                log.append([step, trl, tel, tra, tea, norm])
                if step % 1000 == 0:
                    p = forward(W, x3792)[0].double().softmax(-1)
                    answers[step] = (int(p.argmax()), float(p[16]))
        if step % keep_every == 0 and step <= keep_until:
            snaps[step] = {k: v.detach().clone() for k, v in W.items()}
        if step == steps:
            break
        loss = -forward(W, X[TR]).double().log_softmax(-1).gather(1, Y[TR, None]).mean()
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return W, np.array(log), snaps, answers

t = time.time()
W, LOG, SNAPS, ANSWERS = train(wd=1.0)
print(f"trained 40,000 steps in {time.time() - t:.0f} s")
at = lambda s: LOG[LOG[:, 0] == s][0]
for s in (0, 200, 1400, 4000, 6000, 7500, 8000, 40000):
    r = at(s)
    print(f"step {s:6,}: training sums right {r[3]:6.1%}   test sums right {r[4]:6.1%}   test loss {r[2]:7.3f}   weight size {r[5]:5.1f}")
first = lambda col: int(LOG[LOG[:, col] >= 0.99][0, 0])
check("first step with 99% of training sums right", first(3), LECTURE["step_train99"], 100)
check("first step with 99% of test sums right", first(4), LECTURE["step_test99"], 500)
check("test sums right at step 6,000", round(at(6000)[4], 4), LECTURE["log"]["6000"][3], 0.05)
check("highest test loss (worse than guessing)", round(LOG[:, 2].max(), 2), round(LECTURE["test_loss_peak"], 2), 1.0)
check("largest weight size", round(LOG[:, 5].max(), 2), round(LECTURE["norm_peak"], 2), 0.3)
check("weight size at the end", round(at(40000)[5], 2), round(LECTURE["log"]["40000"][4], 2), 0.3)
check("test sums right at the end", at(40000)[4], LECTURE["log"]["40000"][3])
check("its answer to 37 + 92 at step 4,000", ANSWERS[4000][0], LECTURE["answer_at"]["4000"])
check("its answer to 37 + 92 at step 8,000", ANSWERS[8000][0], LECTURE["answer_at"]["8000"])

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.2))
s = np.maximum(LOG[:, 0], 50)
ax1.plot(s, LOG[:, 3], label="training sums"); ax1.plot(s, LOG[:, 4], ls="--", label="test sums")
ax1.set_xscale("log"); ax1.set_xlabel("training step"); ax1.set_ylabel("right"); ax1.legend(fontsize=8)
ax1.axvspan(6000, 7500, color=INK, alpha=0.08)
ax2.plot(s, LOG[:, 2], label="test loss"); ax2.set_xscale("log"); ax2.set_yscale("log"); ax2.axhline(math.log(P), lw=0.6, ls=":")
ax2.text(60, math.log(P) * 1.3, "guessing", fontsize=8); ax2.set_xlabel("training step"); ax2.set_ylabel("test loss")
plt.show()

**Without weight decay (Plates VII, VIII).** The same network and seed, no pull toward zero, for 10,000 steps: it memorises just as fast and never learns to add. Memorising needs big weights; the clock needs small ones.

In [ ]:
t = time.time()
_, LOG0, _, _ = train(wd=0.0, steps=10000, keep_until=-1)
r0 = LOG0[-1]
print(f"no weight decay, step 10,000 ({time.time() - t:.0f} s): test sums right {r0[4]:.1%}, weight size {r0[5]:.1f}")
check("no weight decay: test sums right at step 10,000", round(r0[4], 4), LECTURE["no_decay_10000"][3], 0.01)
check("no weight decay: weight size at step 10,000", round(r0[5], 2), round(LECTURE["no_decay_10000"][4], 2), 0.5)
fig, ax = plt.subplots(figsize=(6, 2.8))
ax.plot(LOG[:, 0], LOG[:, 5], label="weight decay 1"); ax.plot(LOG0[:, 0], LOG0[:, 5], ls="--", label="no weight decay")
ax.set_xlim(0, 14000); ax.set_xlabel("training step"); ax.set_ylabel("size of all the weights"); ax.legend(fontsize=8)
plt.show()

## Part 4 · Waves over the numbers (Plates IX, X)

Any pattern over the numbers 0 to 112 is a sum of waves: a constant, plus cos(2πkn/113) and sin(2πkn/113) for k = 1 … 56. That is 113 waves for 113 numbers, so the fit is exact. Then we ask how much of the trained network's number table (its 113 rows of 128 numbers) lives at each frequency k.

In [ ]:
n = torch.arange(P, device=DEV, dtype=torch.float64)
rows_ = [torch.ones(P, device=DEV, dtype=torch.float64)]
for k in range(1, P // 2 + 1):
    rows_ += [torch.cos(2 * math.pi * k * n / P), torch.sin(2 * math.pi * k * n / P)]
F_ = torch.stack(rows_)
F_ = (F_ / F_.norm(dim=1, keepdim=True)).float()                   # 113 waves of length 1, all at right angles
pattern = torch.randn(P, generator=torch.Generator().manual_seed(1)).to(DEV)
rebuilt = F_.T @ (F_ @ pattern)
print(f"a random pattern rebuilt from its 113 waves: largest error {float((rebuilt - pattern).abs().max()):.1e}")

def freq_norms(W_E):
    c = (F_ @ W_E[:P]).norm(dim=1) ** 2
    return [float(c[0] ** 0.5)] + [float((c[2 * k - 1] + c[2 * k]) ** 0.5) for k in range(1, P // 2 + 1)]

with torch.no_grad():
    fn = freq_norms(W["W_E"])
top = sorted(range(1, 57), key=lambda k: -fn[k])[:3]
print("the table's largest frequencies:", ", ".join(f"k = {k}: {fn[k]:.2f}" for k in top), f"; every other at most {max(fn[k] for k in range(1, 57) if k not in top):.2f}")
for k in LECTURE["key"]:
    check(f"size of the number table at frequency {k}", round(fn[k], 2), round(LECTURE["embed_norms"][str(k)], 2), 0.1)
fig, ax = plt.subplots(figsize=(7, 2.4))
ax.bar(range(1, 57), fn[1:], color=INK, width=0.7)
ax.set_xlabel("frequency k"); ax.set_ylabel("size"); ax.set_title("the trained number table, frequency by frequency", fontsize=10)
plt.show()

## Part 5 · The clock (Plates XI, XII)

Take the plane of one key frequency, k = 8: the two directions Σ cos(2πka/113)·E[a] and Σ sin(2πka/113)·E[a] in the table's 128-number space. Seen in that plane, the numbers sit on a circle, each at angle 2πka/113, so number a is 8a ticks round (mod 113). Adding is turning: 37 sits 70 ticks round, 92 sits 58; 70 + 58 = 128, which is 15 ticks round, exactly where 16 sits.

In [ ]:
def clock(W_E, k):
    a = torch.arange(P, device=W_E.device, dtype=torch.float32)
    c, s = torch.cos(2 * math.pi * k * a / P), torch.sin(2 * math.pi * k * a / P)
    E_ = W_E[:P]
    u = c @ E_; u = u / u.norm()
    v = s @ E_; v = v - (v @ u) * u; v = v / v.norm()
    x_, y_ = E_ @ u, E_ @ v
    ang = torch.atan2(y_, x_)
    pred = 2 * math.pi * k * a / P
    fit = max(float(torch.cos(ang - pred).mean()), float(torch.cos(ang + pred).mean()))
    return x_.cpu().numpy(), y_.cpu().numpy(), fit

with torch.no_grad():
    clocks = {k: clock(W["W_E"], k) for k in LECTURE["key"]}
for k, (_, _, fit) in clocks.items():
    print(f"frequency {k:2d}: the numbers sit at their predicted angles, fit {fit:.4f}")
    check(f"clock fit at frequency {k}", round(fit, 4), LECTURE["clock_fit"][str(k)], 0.002)
print("ticks round on the frequency-8 clock: 37 ->", 8 * 37 % P, ", 92 ->", 8 * 92 % P, ", together", 8 * 37 % P + 8 * 92 % P,
      "->", (8 * 37 % P + 8 * 92 % P) % P, "= where 16 sits:", 8 * 16 % P)
fig, ax = plt.subplots(figsize=(3.8, 3.8))
x_, y_, _ = clocks[8]
ax.scatter(x_, y_, s=8, color=INK)
for m in (37, 92, 16):
    ax.annotate(str(m), (x_[m], y_[m]), (6, 4), textcoords="offset points", fontsize=9)
ax.set_aspect("equal"); ax.axis("off"); ax.set_title("the number table, seen along frequency 8", fontsize=10)
plt.show()

## Part 6 · One neuron, two waves (Plates XIII, XIV)

Neuron 27, for every sum: a down, b across. Its raw map is a checkerboard. A neuron adds a wave in a to a wave in b and applies ReLU, so most of its activity is "a wave in a plus a wave in b"; the part that depends on a + b alone (the stripes) is the product the ReLU creates, and it is what the scores read.

In [ ]:
keep = {}
with torch.no_grad():
    logits_all = forward(W, X, keep).double()
POST = keep["post"].reshape(P, P, -1)                                  # [a, b, neuron]

def shares(g):
    """Share of a map's variance explained by a function of (a + b) alone, and by f(a) + g(b)."""
    g = g.double(); m = g.mean()
    idx = (torch.arange(P, device=g.device)[:, None] + torch.arange(P, device=g.device)[None, :]) % P
    diag = torch.zeros(P, device=g.device, dtype=g.dtype).index_add_(0, idx.flatten(), g.flatten()) / P
    add = g.mean(1, keepdim=True) + g.mean(0, keepdim=True) - m
    ss = ((g - m) ** 2).sum()
    return float(1 - ((g - diag[idx]) ** 2).sum() / ss), float(1 - ((g - add) ** 2).sum() / ss)

s_sum, s_add = shares(POST[:, :, 27])
print(f"neuron 27: a function of a + b explains {s_sum:.1%}; a wave in a plus a wave in b explains {s_add:.1%}")
check("neuron 27, the part that depends on a + b", round(s_sum, 4), LECTURE["neuron_shares"]["27"][0], 0.01)
check("neuron 27, a wave in a plus a wave in b", round(s_add, 4), LECTURE["neuron_shares"]["27"][1], 0.01)

g27 = POST[:, :, 27]
idx = (torch.arange(P, device=DEV)[:, None] + torch.arange(P, device=DEV)[None, :]) % P
diag = torch.zeros(P, device=DEV).index_add_(0, idx.flatten(), g27.flatten()) / P
fig, axs = plt.subplots(1, 2, figsize=(7, 3.4))
axs[0].imshow(g27.cpu().numpy(), cmap="Greys"); axs[0].set_title("neuron 27: every sum", fontsize=10)
axs[1].imshow(diag[idx].cpu().numpy(), cmap="Greys"); axs[1].set_title("its part that depends on a + b", fontsize=10)
for ax in axs:
    ax.set_xlabel("b"); ax.set_ylabel("a"); ax.set_xticks([]); ax.set_yticks([])
plt.show()

**Every neuron has a frequency (Plate XIV).** For each neuron, measure how much of its map's variance lives at each frequency (counting waves in a, in b and in a·b). Each neuron has one main frequency, and every one is a key frequency.

In [ ]:
A_ = POST - POST.mean((0, 1), keepdim=True)
Ah = torch.einsum("ia,abn,jb->ijn", F_, A_, F_) ** 2                  # each neuron's map, in waves of a and of b
tot = Ah.sum((0, 1)) + 1e-12
energy = torch.stack([Ah[[0, 2 * k - 1, 2 * k]][:, [0, 2 * k - 1, 2 * k]].sum((0, 1)) for k in range(1, 57)])
nfreq = (energy.argmax(0) + 1).cpu().numpy()
purity = (energy.max(0).values / tot).cpu().numpy()
counts = np.bincount(nfreq, minlength=57)
key = [int(k) for k in np.nonzero(counts >= 8)[0]]
print("key frequencies (8 or more neurons each):", key, "with", [int(counts[k]) for k in key], "neurons;",
      f"median purity {np.median(purity):.1%}")
check("key frequencies", key, LECTURE["key"])
for k, n_ in zip(key, LECTURE["neuron_counts"]):
    check(f"neurons at frequency {k}", int(counts[k]), n_, 8)
check("median share of a neuron's activity in its own frequency", round(float(np.median(purity)), 4), LECTURE["purity_median"], 0.01)
check("neuron 27's frequency", int(nfreq[27]), LECTURE["neuron_freq"]["27"])

## Part 7 · The vote (Plate XV)

How does a clock become an answer? With each sum's average score removed, the scores are waves in (a + b − c) over the candidate answers c. Keep only one key frequency's wave at a time: each peaks exactly at c = 16 for 37 + 92, and added up they tower over every wrong answer.

In [ ]:
Lg = logits_all.reshape(P, P, P); Lg = Lg - Lg.mean(-1, keepdim=True)
av = torch.arange(P, device=DEV, dtype=torch.float64)
sgrid = av[:, None, None] + av[None, :, None] - av[None, None, :]      # a + b - c

def wave_part(Lg, k):
    comp = torch.zeros_like(Lg)
    for f_ in (torch.cos, torch.sin):
        D = f_(2 * math.pi * k * sgrid / P); D = D - D.mean(-1, keepdim=True)
        comp += (Lg * D).sum() / (D * D).sum() * D
    return comp

waves = {k: wave_part(Lg, k)[37, 92] for k in key}
total = sum(waves.values())
for k, w in waves.items():
    print(f"frequency {k:2d}: its wave peaks at c = {int(w.argmax())}, giving 16 {float(w[16]):+.1f}")
    check(f"frequency {k}'s vote for 16", round(float(w[16]), 2), round(LECTURE["waves_at_16"][str(k)], 2), 0.15)
best_wrong = float(torch.cat([total[:16], total[17:]]).max())
print(f"together: 16 gets {float(total[16]):.1f}, the best wrong answer {best_wrong:.1f}")
check("the vote for 16, all waves", round(float(total[16]), 2), LECTURE["vote_16"], 0.3)
check("the best wrong answer's vote", round(best_wrong, 2), LECTURE["vote_best_wrong"], 0.3)
real = logits_all.reshape(P, P, P)[37, 92]
check("the real score for 16", round(float(real[16]), 2), LECTURE["logit_16"], 0.3)
check("neuron 182 fires for 37 + 92", round(float(POST[37, 92, 182]), 3), LECTURE["neuron_182_fires"], 0.05)
fig, ax = plt.subplots(figsize=(7, 2.8))
for (k, w), ls in zip(waves.items(), (":", "--", "-.")):
    ax.plot(w.cpu().numpy(), ls=ls, lw=1, color=INK2, label=f"frequency {k}")
ax.plot(total.cpu().numpy(), lw=2, label="the sum"); ax.axvline(16, lw=0.6, ls=":")
ax.set_xlabel("candidate answer c"); ax.legend(fontsize=8, ncol=4, loc="lower center", bbox_to_anchor=(0.5, 1.0))
plt.show()

## Part 8 · Grokking from the inside (Plate XVI)

Two *progress measures* (Nanda et al. 2023), computed from the weights kept every 500 steps:
- **restricted test loss**: the scores with only the key-frequency waves kept, on the test sums. Low means the clock alone already gets the exam right.
- **excluded training loss**: the scores with those waves taken out, on the training sums. High means the homework is now carried by the clock.

The clock alone gets the hidden sums nearly right long before the test score jumps.

In [ ]:
tr_mask = is_train.reshape(P, P).to(DEV)
def ce(Lg_, mask):
    lp = Lg_.log_softmax(-1).gather(-1, Y.reshape(P, P)[..., None])[..., 0]
    return float(-lp[mask].mean())
prog = []
for step, Ws in sorted(SNAPS.items()):
    with torch.no_grad():
        Lg_ = forward(Ws, X).double().reshape(P, P, P)
    Lg_ = Lg_ - Lg_.mean(-1, keepdim=True)
    comp = sum(wave_part(Lg_, k) for k in key)
    prog.append((step, ce(Lg_, ~tr_mask), ce(comp, ~tr_mask), ce(Lg_ - comp, tr_mask)))
prog = np.array(prog)
for s in (2000, 4000, 6000, 8000):
    r = prog[prog[:, 0] == s][0]
    print(f"step {s:5,}: test loss {r[1]:8.4f}   restricted test loss {r[2]:7.4f}   excluded training loss {r[3]:6.2f}")
r6 = prog[prog[:, 0] == 6000][0]
check("test loss at step 6,000", round(r6[1], 2), round(LECTURE["progress"]["6000"]["test_loss"], 2), 0.3)
check("restricted test loss at step 6,000", round(r6[2], 4), LECTURE["progress"]["6000"]["restricted_test"], 0.004)
check("restricted test loss at step 4,000", round(prog[prog[:, 0] == 4000][0][2], 3), round(LECTURE["progress"]["4000"]["restricted_test"], 3), 0.05)
fig, ax = plt.subplots(figsize=(6.5, 3))
ax.plot(prog[:, 0], prog[:, 1], label="test loss"); ax.plot(prog[:, 0], prog[:, 2], ls="--", label="restricted test loss (clock only)")
ax.plot(prog[:, 0], prog[:, 3], ls=":", label="excluded training loss")
ax.set_yscale("log"); ax.set_xlabel("training step"); ax.legend(fontsize=8, loc="lower left")
ax.axvspan(6000, 7500, color=INK, alpha=0.08)
plt.show()
print(f"total run time: {(time.time() - T0_START) / 60:.1f} min")

## Your turn

1. **More weight decay.** Train with weight decay 3 for 4,000 steps: when does it first get 99% of the test sums right? *Hint: `train(wd=3.0, steps=4000, keep_until=-1)`; the lecture's sweep found 1,800.*
2. **Another clock.** On the frequency-21 clock, where do 37, 92 and 16 sit (in ticks)? Check that the turns add up. *Hint: ticks = 21·n mod 113.*
3. **Another neuron.** Neuron 182 belongs to frequency 1. Measure its two shares as in Part 6. *Hint: `shares(POST[:, :, 182])`.*
4. **What did it answer, and when?** Print the network's answer to 37 + 92 every 1,000 steps (`ANSWERS`). When did it stop answering 0?

In [ ]:
# 1. TODO: weight decay 3

In [ ]:
# 2. TODO: the frequency-21 clock

In [ ]:
# 3. TODO: neuron 182

In [ ]:
# 4. TODO: the answer to 37 + 92, step by step

## Solutions

In [ ]:
# 1. More weight decay groks sooner
_, LOG3, _, _ = train(wd=3.0, steps=4000, keep_until=-1)
s99 = int(LOG3[LOG3[:, 4] >= 0.99][0, 0])
print(f"weight decay 3: 99% of test sums right first at step {s99:,}")
check("weight decay 3: first step with 99% of test sums right", s99, LECTURE["grok_step_wd3"], 300)

In [ ]:
# 2. The frequency-21 clock
t37, t92, t16 = (21 * m % P for m in (37, 92, 16))
print(f"37 sits {t37} ticks round, 92 sits {t92}; {t37} + {t92} = {t37 + t92} -> {(t37 + t92) % P}, where 16 sits ({t16})")

In [ ]:
# 3. Neuron 182
s_sum, s_add = shares(POST[:, :, 182])
print(f"neuron 182 (frequency {nfreq[182]}): a + b alone {s_sum:.1%}, a wave in a plus a wave in b {s_add:.1%}")
check("neuron 182, a wave in a plus a wave in b", round(s_add, 4), LECTURE["neuron_shares"]["182"][1], 0.01)

In [ ]:
# 4. The answer to 37 + 92, step by step
for s, (ans, p16) in sorted(ANSWERS.items())[:11]:
    print(f"step {s:6,}: answers {ans:3d}   chance on 16 {p16:.4f}")